In [ ]:
import json
import random

random.seed(42)

dataset_with_answers_path = "musique_ans_v1.0_dev.jsonl"
examples_with_answers = []

all_lines = []
with open(dataset_with_answers_path, 'r', encoding='utf-8') as f:
    all_lines = [json.loads(line) for line in f]


examples_with_answers = random.sample(all_lines, 700)[300:400]

print(f"Загружено {len(examples_with_answers)} случайных примеров")

Загружено 100 случайных примеров


In [ ]:
import pandas as pd
from typing import List, Dict, Any
import re

def transform_dataset(dataset: List[Dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Преобразует датасет в два DataFrame:
    1. paragraphs_df - параграфы с абсолютной нумерацией
    2. questions_df - подвопросы с абсолютными ID параграфов и ID общего вопроса

    В тексте вопросов заменяет #1, #2, #3 и т.д. на ответы соответствующих подвопросов.
    """
    paragraphs_list = []
    questions_list = []
    global_paragraph_counter = 1
    paragraph_mapping = {}

    for example in dataset:
        example_id = example['id']

        for paragraph in example['paragraphs']:
            paragraph_idx = paragraph['idx']
            key = (example_id, paragraph_idx)


            paragraph_mapping[key] = global_paragraph_counter


            paragraphs_list.append({
                'absolute_id': global_paragraph_counter,
                'text': paragraph['paragraph_text']
            })

            global_paragraph_counter += 1


    for example in dataset:
        example_id = example['id']
        main_question_id = example['id']
        main_question_text = example['question']

        decomposition = example.get('question_decomposition', [])


        processed_questions = []

        for i, sub_question in enumerate(decomposition):
            question_text = sub_question['question']


            answer_mapping = {}
            for j in range(i):
                if j < len(processed_questions):
                    placeholder = f"#{j+1}"
                    answer_mapping[placeholder] = decomposition[j].get('answer', '')


            for placeholder, answer in answer_mapping.items():
                if answer:

                    #print(question_text)
                    pattern = re.escape(placeholder)
                    #print(pattern)

                    question_text = re.sub(placeholder, answer, question_text)
                    #print(question_text)

            processed_questions.append(question_text)


        for i, sub_question in enumerate(decomposition):
            paragraph_support_idx = sub_question.get('paragraph_support_idx')


            paragraph_absolute_id = None
            if paragraph_support_idx is not None:
                key = (example_id, paragraph_support_idx)
                paragraph_absolute_id = paragraph_mapping.get(key)

            questions_list.append({
                'question': processed_questions[i],
                'paragraph_absolute_id': paragraph_absolute_id,
                'main_question_id': main_question_id,
                'main_question_text': main_question_text
            })


    paragraphs_df = pd.DataFrame(paragraphs_list)
    questions_df = pd.DataFrame(questions_list)

    return paragraphs_df, questions_df

In [ ]:
paragraphs_df, questions_df = transform_dataset(examples_with_answers)
paragraphs_df

,absolute_id,text
0,1,Liam Thomas Garrigan (born 17 October 1981) is...
1,2,Ideas for a Conan film were proposed as early ...
2,3,"Jeffrey Shawn Swords (born December 27, 1973 i..."
3,4,``Born in the U.S.A. ''is a 1984 song written ...
4,5,"Haji Sahib of Turangzai, the most famous Pukht..."
...,...,...
1993,1994,"In the earlier seasons of Family Guy, Clevelan..."
1994,1995,Lacey Chabert voiced Meg for the first product...
1995,1996,Meg Griffin Family Guy character First appeara...
1996,1997,John Herbert Family Guy character First appear...


In [ ]:
import random
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

def create_retriever(qa_df, paragraphs_df, top_k=5, p_correct=0.5, seed=None, model_name='all-MiniLM-L6-v2', device='cuda'):
    """
    Создаёт функцию-имитатор ретривера с использованием эмбеддингов all-MiniLM-L6-v2.
    При КАЖДОМ вызове возвращает новую порцию документов.
    Правильный документ появляется в выдаче с вероятностью p_correct.
    Если правильный документ естественным образом попал в топ-k, но вероятность не сработала,
    он удаляется и заменяется следующим по релевантности неправильным.

    Параметры:
        qa_df : pd.DataFrame
            Датафрейм с колонками 'question' (текст подвопроса) и 'paragraph_absolute_id' (id правильного параграфа).
        paragraphs_df : pd.DataFrame
            Датафрейм с колонками 'id' (уникальный идентификатор) и 'text' (текст параграфа).
        top_k : int
            Количество параграфов, возвращаемых за один вызов.
        p_correct : float
            Вероятность того, что в возвращаемой порции окажется правильный документ.
        seed : int, optional
            Для воспроизводимости случайных выборов.
        model_name : str
            Название модели sentence-transformers.
        device : str
            Устройство для вычислений ('cuda' или 'cpu').

    Возвращает:
        function
            Функция retrieve(query: str) -> dict
            С полями:
                'docs': список строк вида "id. текст"
                'has_correct': bool, есть ли среди выданных документов правильный для этого вопроса
    """
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    # Загружаем модель эмбеддингов
    encoder = SentenceTransformer(model_name, device=device)

    # Предвычисляем эмбеддинги всех параграфов
    paragraph_ids = paragraphs_df['absolute_id'].tolist()
    paragraph_texts = paragraphs_df['text'].tolist()
    print("Вычисление эмбеддингов параграфов...")
    paragraph_embeddings = encoder.encode(paragraph_texts, convert_to_tensor=True, device=device, show_progress_bar=True)
    paragraph_embeddings = torch.nn.functional.normalize(paragraph_embeddings, p=2, dim=1)

    # Словарь правильных id для каждого вопроса
    correct_map = {}
    for _, row in qa_df.iterrows():
        q = row['question']
        pid = row['paragraph_absolute_id']
        correct_map.setdefault(q, set()).add(pid)

    # Словарь для быстрого доступа к тексту
    text_by_id = dict(zip(paragraph_ids, paragraph_texts))

    def retrieve(query, last):
        # Вычисляем эмбеддинг запроса
        query_emb = encoder.encode(query, convert_to_tensor=True, device=device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)

        # Косинусное сходство со всеми параграфами
        similarities = torch.matmul(paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [paragraph_ids[i] for i in sorted_indices]

        correct_ids = correct_map.get(query, set())
        #correct_ids = set(questions_df[qa_df['question'] == query]['paragraph_absolute_id'])

        # Базовый топ-k по релевантности
        candidates = sorted_ids[:top_k]
        #print('candidates',candidates)
        #print('correct_ids',correct_ids)
        if last == False:
        # Решаем, должен ли правильный присутствовать в выдаче
          include_correct = random.random() < p_correct
        else:
          include_correct = True
        #print("include_correct",include_correct)

        # Правильные id, которые уже есть в candidates
        correct_in_candidates = [pid for pid in candidates if pid in correct_ids]
        #print(correct_in_candidates)

        if include_correct:
            # Хотим, чтобы правильный был в выдаче
            if not correct_in_candidates:
                # Если правильного нет, добавляем один случайный правильный, заменяя случайный элемент
                if correct_ids:
                    chosen_correct = random.choice(list(correct_ids))
                    # Заменяем случайный элемент в candidates
                    replace_idx = random.randint(0, len(candidates)-1)
                    candidates[replace_idx] = chosen_correct
            # else: правильный уже есть, ничего не делаем (оставляем как есть)
        else:
            # Не хотим правильного в выдаче
            if correct_in_candidates:
                # Удаляем все правильные из candidates
                candidates_without_correct = [pid for pid in candidates if pid not in correct_ids]
                # Добираем недостающие из следующих по релевантности (только неправильные)
                # Ищем следующие id в sorted_ids, которые не в correct_ids и ещё не в candidates
                next_ids = []
                for pid in sorted_ids[top_k:]:
                    if pid not in correct_ids and pid not in candidates_without_correct:
                        next_ids.append(pid)
                    if len(candidates_without_correct) + len(next_ids) >= top_k:
                        break
                # Дополняем до top_k
                candidates = (candidates_without_correct + next_ids)[:top_k]
            # else: правильного и так нет, оставляем candidates как есть

        # Перемешиваем, чтобы правильный не всегда был на одной позиции
        random.shuffle(candidates)

        # Формируем вывод
        docs = [f"{pid}. {text_by_id[pid]}" for pid in candidates]
        ids = [pid for pid in candidates]

        return {
            'docs': docs,
            'ids': ids
        }

    return retrieve

/root/qwen_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TOP_K = 5
P_CORRECT = 0.3 #0.3
SEED = 42
RETRIEVER_MODEL = 'all-MiniLM-L6-v2'
DEVICE = "cuda"

retriever = create_retriever(questions_df, paragraphs_df, top_k=TOP_K, p_correct=P_CORRECT,
                             seed=SEED, model_name=RETRIEVER_MODEL, device=DEVICE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2530.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Вычисление эмбеддингов параграфов...


Batches: 100%|██████████| 63/63 [00:01<00:00, 40.69it/s]


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "INDEX_SEARCH_TOOL",
            "description": "Retrieve documents from the index by a search query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": "Submit the final answer with the document ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Document ID that best answers the question"}
                },
                "required": ["id"]
            }
        }
    }
]

In [ ]:
import json
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Any, Optional, Tuple, Union


class QwenAgent:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-7B-Instruct", device_map: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map=device_map
        )
        self.model.eval()

    def _call_model(self, messages: List[Dict[str, Any]], max_new_tokens: int = 512):
        """
        Вызывает модель и возвращает сгенерированный текст, ID токенов и логиты.
        Returns:
            (response, generated_ids, scores)
            response: str
            generated_ids: torch.Tensor (1, num_generated)
            scores: tuple of torch.Tensor (num_generated, 1, vocab_size)
        """
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tools=TOOLS,
            add_generation_prompt=True,
            tokenize=False
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=True,
            )
        # generated_ids: [1, total_seq_len]; входные + новые
        total_ids = outputs.sequences[0]
        input_len = inputs.input_ids.shape[1]
        generated_ids = total_ids[input_len:]  # только новые токены
        scores = outputs.scores  # tuple of tensors, каждый [1, vocab_size]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=False)
        return response, generated_ids, scores

    def _extract_tool_call(self, response: str) -> Optional[Dict[str, Any]]:
        # 1. Пробуем JSON
        json_pattern = r'<tool_call>(.*?)</tool_call>'
        match = re.search(json_pattern, response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

        # 2. Пробуем XML: <function=NAME><parameter=ARG>value</parameter></function>
        xml_pattern = r'<tool_call>.*?<function=(\w+)>.*?<parameter=(\w+)>(.*?)</parameter>.*?</tool_call>'
        match = re.search(xml_pattern, response, re.DOTALL)
        if match:
            name = match.group(1)
            param_name = match.group(2)
            param_value = match.group(3).strip()
            # Убираем возможные кавычки вокруг строки
            if param_value.startswith('"') and param_value.endswith('"'):
                param_value = param_value[1:-1]
            return {"name": name, "arguments": {param_name: param_value}}

        return None

    def _compute_tool_metric(self, response: str, generated_ids: torch.Tensor, scores: tuple, metric: str = 'entropy') -> Optional[float]:
        """
        Вычисляет среднюю энтропию (или NLL) для токенов, образующих вызов инструмента.
        Аргументы:
            response: полный сгенерированный текст (без входа)
            generated_ids: тензор ID сгенерированных токенов (1, num_tokens)
            scores: кортеж тензоров логитов для каждого шага
            metric: 'entropy' или 'nll'
        Returns:
            среднее значение метрики по токенам вызова, или None, если вызов не найден
        """
        # Ищем в ответе подстроку вызова инструмента (включая теги)
        tool_call_pattern = r'(<tool_call>.*?</tool_call>)'
        match = re.search(tool_call_pattern, response, re.DOTALL)
        if not match:
            return None
        tool_call_str = match.group(1)
        start_idx = match.start()
        end_idx = match.end()

        # Декодируем каждый токен по отдельности для позиций
        tokens = []
        positions = []
        pos = 0
        for tid in generated_ids:
            token_str = self.tokenizer.decode([tid], skip_special_tokens=False)
            tokens.append(token_str)
            positions.append(pos)
            pos += len(token_str)

        full_text = ''.join(tokens)


        # Определяем индексы токенов, перекрывающихся с tool_call_str
        indices = []
        for i, (token, p) in enumerate(zip(tokens, positions)):
            token_end = p + len(token)
            if token_end > start_idx and p < end_idx:
                indices.append(i)

        if not indices:
            return None

        values = []
        for i in indices:
            logits = scores[i]          # [1, vocab_size]
            logp = F.log_softmax(logits, dim=-1)   # [1, vocab_size]
            if metric == 'entropy':
                p = logp.exp()
                entropy = -(p * logp).sum(dim=-1).item()
                values.append(entropy)
            elif metric == 'nll':
                token_id = generated_ids[i].item()
                nll = -logp[0, token_id].item()
                values.append(nll)
            else:
                raise ValueError("metric must be 'entropy' or 'nll'")

        return sum(values) / len(values)

    def step(self, messages: List[Dict[str, Any]], metric: str = 'entropy') -> Tuple[Optional[Dict[str, Any]], str, Optional[float]]:
        """
        Выполняет один шаг: вызывает модель, извлекает вызов инструмента и вычисляет метрику для токенов вызова.
        Args:
            messages: история сообщений
            metric: 'entropy' или 'nll' – тип вычисляемой метрики
        Returns:
            (tool_call, raw_response, metric_value)
            tool_call: словарь с name и arguments, если найден, иначе None
            raw_response: полный текст ответа модели
            metric_value: средняя энтропия (или NLL) для токенов вызова, или None, если вызов не найден
        """
        #print('hello')
        response, gen_ids, scores = self._call_model(messages)
        tool_call = self._extract_tool_call(response)
        tool_metric = self._compute_tool_metric(response, gen_ids, scores, metric) if tool_call else None
        return tool_call, response, tool_metric

In [ ]:
import re
import torch
import torch.nn.functional as F

# ======================== ПРОМПТ АГЕНТА ========================
SYSTEM_PROMPT = """
ROLE:
You are a precise knowledge assistant. Answer the user's question using the provided documents.

RULES:
1. If documents contain the answer, select the most explicit and complete document and call `submit_answer` with its id.
2. If no document clearly answers the question, call `INDEX_SEARCH_TOOL` to retrieve more relevant documents. You may reformulate the query.
3. Do not call `INDEX_SEARCH_TOOL` repeatedly with semantically identical queries.
4. Never invent document ids or content. Only rely on tool-provided observations.

FORMAT:
- Before each tool call, you may provide a short thought (1‑2 sentences, <40 words) inside `<thought>` tags.
- Your final response must be a tool call in the following format:
  `<tool_call>{"name": "<tool_name>", "arguments": {...}}</tool_call>`
- Do not output `<observation>` or document text; the system will provide them after your tool call.

EXAMPLES:

EXAMPLE 1 (answer not in initial documents):
question: Who is the current president of France?
Documents:
31. The French President is elected for a five-year term.
222. The Élysée Palace is the official residence of the President.
413. France has a semi-presidential system.
<thought>These documents give background but don't name the current president. I need a more specific query.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Who is the current president of France"}}</tool_call>
<observation> (system provides)
6. Emmanuel Macron was re-elected in 2022.
79. Emmanuel Macron is the current president of France.
</observation>
<thought>Document 79 explicitly names Emmanuel Macron as current president.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 79}}</tool_call>

EXAMPLE 2 (answer found after tool call):
question: When did the Berlin Wall fall?
Documents:
45. The Berlin Wall divided Berlin from 1961 to 1989.
12. The fall paved the way for German reunification.
<thought>The exact date is missing. I'll search for it.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Berlin Wall fall date"}}</tool_call>
<observation>
23. The Berlin Wall fell on November 9, 1989.
34. November 9, 1989 is a historic date.
</observation>
<thought>Document 23 gives the exact date.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 23}}</tool_call>

Now begin."""

In [ ]:
llm = QwenAgent(model_name="Qwen/Qwen3.5-4B")

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 10242.50it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:02<00:00, 186.56it/s]


In [ ]:
def index_search_tool(query, last):
    result = retriever(query, last)   # result['docs'] — список строк документов
    tool_content = "\n".join(result['docs'])   # просто текст, без обёртки
    return result['ids'], tool_content

In [ ]:
THINK_AGAIN_PROMPT = (
    "Before calling INDEX_SEARCH_TOOL, re‑examine ALL provided documents. "
    "If the answer is present (even indirectly) or can be derived from them, "
    "DO NOT call the tool – answer immediately from the documents. "
    "Only call INDEX_SEARCH_TOOL if you are absolutely certain the required information is missing. "
    "Never guess or invent an answer. When uncertain, always prefer calling the tool over guessing."
)


In [ ]:
def run_agent_for_subquestion(agent, subquestion, initial_docs, ids, correct_ids, max_calls=5, max_steps=5, entropy_threshold=0.0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"question: {subquestion}\n\nDocuments:\n{initial_docs}"}
    ]
    tool_calls_used = 0
    step_info_history = []
    final_answer_id = None
    success = False

    for step in range(max_steps):
        tool_call, response, metric = agent.step(messages, metric='entropy')
        #print(f'Step {step+1} response:\n{response}')
        if metric is not None:
            print(f"Tool call metric (entropy): {metric:.4f}")

        # --- Intervention: low‑entropy tool call → remind to think again ---
        if (entropy_threshold > 0 and metric is not None and metric > entropy_threshold
                and tool_call and tool_call.get("name") == "INDEX_SEARCH_TOOL"):
            print(f"Metric {metric:.4f} below threshold {entropy_threshold}, injecting think-again prompt.")
            messages.append({"role": "user", "content": THINK_AGAIN_PROMPT})
            tool_calls_used += 1

            step_info_history.append({
                "step": step,
                "action": "THINK_AGAIN_PROMPT",
                "has_correct": len(set(ids) & correct_ids) > 0,
                "entropy": metric,
                "text": response,
            })

            if tool_calls_used >= max_calls:
                print("Max tool calls exceeded, stopping.")
                break
            continue  # skip normal processing for this step

        # --- Normal processing (no intervention) ---
        if not tool_call:
            print("No valid tool call found. Stopping.")
            break

        name = tool_call.get("name")
        args = tool_call.get("arguments", {})

        has_correct = len(set(ids) & correct_ids) > 0

        step_info = {
            "step": step,
            "action": f"{name}({json.dumps(args)})",
            "has_correct": has_correct,
            "entropy": metric,
            "text": response,
        }
        step_info_history.append(step_info)

        messages.append({"role": "assistant", "content": response})

        if name == "submit_answer":
            final_answer_id = args.get("id")
            success = True
            break

        elif name == "INDEX_SEARCH_TOOL":
            if tool_calls_used >= max_calls:
                print("Max tool calls exceeded, stopping.")
                break
            query = args.get("query", "")
            last = (step == max_steps - 1)
            ids, tool_content = index_search_tool(query, last)
            has_correct = len(set(ids) & set(correct_ids)) > 0
            #print(f"Tool result (has_correct={has_correct}):\n{tool_content}")
            messages.append({"role": "tool", "name": name, "content": tool_content})
            tool_calls_used += 1
            continue
        else:
            print(f"Unknown tool: {name}")
            break

    if final_answer_id is None:
        final_answer_id = 'No answer'
    return final_answer_id, step_info_history, success

In [ ]:
def make_serializable(obj):
    if isinstance(obj, set):
        return list(obj)
    if hasattr(obj, 'tolist'):   # для тензоров PyTorch / TensorFlow / NumPy
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(item) for item in obj]
    return obj

In [ ]:
from tqdm import tqdm
import numpy as np
import json

unique_main_ids = questions_df['main_question_id']  # .unique()
results_by_main = {}

MAX_TOOL_CALLS = 5
ENTROPY_THRESHOLD = 0.08245324108860731
SAVE_PATH = 'results_qwen_4B_ENTROPY_THRESHOLD_1.json'



for main_id in tqdm(unique_main_ids, desc="Main questions"):
    #print(f"\n{'='*50}")
    #print(f"Обработка основного вопроса ID: {main_id}")

    subquestions_df = questions_df[questions_df['main_question_id'] == main_id]
    subquestions = subquestions_df['question'].tolist()

    results_for_main = {}
    all_sub_correct = True

    for subq in tqdm(subquestions, desc=f"Подвопросы main_{main_id}", leave=False):
        #print(f"\n  Подвопрос: {subq}")

        first_page = retriever(subq, False)
        ids = first_page['ids']
        initial_docs_str = "\n".join(first_page['docs'])
        correct_ids = set(subquestions_df[subquestions_df['question'] == subq]['paragraph_absolute_id'])

        has_correct = len(set(ids) & correct_ids) > 0
        #print("  Начальные документы (есть правильный? {}):".format(has_correct))
        #print(initial_docs_str)


        doc_id, logits_hist, success = run_agent_for_subquestion(
            llm,
            subq,
            initial_docs_str,
            ids,
            correct_ids,
            max_calls=MAX_TOOL_CALLS,
            entropy_threshold=ENTROPY_THRESHOLD
        )
        try:
            doc_id = int(doc_id)
        except:
            doc_id = doc_id

        is_correct = doc_id in correct_ids
        if not is_correct:
            all_sub_correct = False

        results_for_main[subq] = {
            "found_doc": doc_id,
            "correct_ids": list(correct_ids),
            "is_correct": is_correct,
            "logits_history": logits_hist,
            "initial_has_correct": has_correct,
            "success": success
        }


        #print(f"  Агент вернул: {doc_id} (правильные: {correct_ids}) -> {'✓' if is_correct else '✗'}")

    results_by_main[main_id] = {
        "subquestions": results_for_main,
        "main_correct": all_sub_correct,
        "total_subquestions": len(subquestions),
        "correct_subquestions": sum(1 for r in results_for_main.values() if r['is_correct'])
    }

    correct_count = results_by_main[main_id]["correct_subquestions"]
    total = results_by_main[main_id]["total_subquestions"]
    #print(f"\n  Итоги по основному вопросу {main_id}: правильно {correct_count}/{total} ({correct_count/total*100:.1f}%)")
    #print(f"  Основной вопрос в целом {'✓ верно' if all_sub_correct else '✗ неверно'}")

    serializable_results = make_serializable(results_by_main)
    with open(SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(serializable_results, f, ensure_ascii=False, indent=2)
    #print(f"  Промежуточные результаты сохранены: {SAVE_PATH}")

Main questions:   0%|          | 0/246 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Tool call metric (entropy): 0.0983
Metric 0.0983 below threshold 0.08245324108860731, injecting think-again prompt.


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Tool call metric (entropy): 0.0966
Metric 0.0966 below threshold 0.08245324108860731, injecting think-again prompt.


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

Tool call metric (entropy): 0.0172


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Tool call metric (entropy): 0.0182


Main questions: 100%|██████████| 246/246 [8:11:35<00:00, 119.90s/it]

Tool call metric (entropy): 0.0138


In [ ]:
with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    daresults_by_mainta = json.load(f)

In [ ]:
import numpy as np

def compute_metrics(results_by_main):
    total_subq = 0
    correct_subq = 0
    success_subq = 0
    steps_per_subq = []
    initial_has_correct_stats = {'total': 0, 'correct': 0}
    initial_no_correct_stats = {'total': 0, 'correct': 0}

    main_stats = {}

    for main_id, main_data in results_by_main.items():
        # Проверяем, что main_data - словарь и содержит 'subquestions'
        if not isinstance(main_data, dict):
            print(f"Предупреждение: для main_id {main_id} данные не словарь ({type(main_data)}), пропускаем.")
            continue

        subqs = main_data.get('subquestions', {})
        if not isinstance(subqs, dict):
            print(f"Предупреждение: для main_id {main_id} subquestions не словарь ({type(subqs)}), пропускаем.")
            continue

        num_subq = len(subqs)
        num_correct_main = 0

        for subq, res in subqs.items():
            if not isinstance(res, dict):
                print(f"Предупреждение: для подвопроса {subq} данные не словарь ({type(res)}), пропускаем.")
                continue

            total_subq += 1

            # Определяем правильность ответа
            is_correct = res.get('is_correct', False)
            if is_correct:
                correct_subq += 1
                num_correct_main += 1

            # Успешность завершения
            if res.get('success', False):
                success_subq += 1

            # Количество шагов
            steps = len(res.get('logits_history', []))
            steps_per_subq.append(steps)

            # Статистика по начальной выдаче
            initial_has_correct = res.get('initial_has_correct', False)
            if initial_has_correct:
                initial_has_correct_stats['total'] += 1
                if is_correct:
                    initial_has_correct_stats['correct'] += 1
            else:
                initial_no_correct_stats['total'] += 1
                if is_correct:
                    initial_no_correct_stats['correct'] += 1

        main_stats[main_id] = {
            'num_subq': num_subq,
            'num_correct': num_correct_main,
            'all_correct': num_correct_main == num_subq,
            'main_correct': main_data.get('main_correct', False)  # можно добавить для сравнения
        }

    total_main = len(main_stats)
    main_all_correct = sum(1 for v in main_stats.values() if v['all_correct'])

    metrics = {
        'total_subquestions': total_subq,
        'subq_accuracy': correct_subq / total_subq if total_subq else 0,
        'subq_success_rate': success_subq / total_subq if total_subq else 0,
        'avg_steps_per_subq': np.mean(steps_per_subq) if steps_per_subq else 0,
        'std_steps_per_subq': np.std(steps_per_subq) if steps_per_subq else 0,
        'min_steps_per_subq': min(steps_per_subq) if steps_per_subq else 0,
        'max_steps_per_subq': max(steps_per_subq) if steps_per_subq else 0,
        'total_main_questions': total_main,
        'main_accuracy': main_all_correct / total_main if total_main else 0,
        'main_all_correct_count': main_all_correct,
        'subq_accuracy_when_initial_has_correct': (
            initial_has_correct_stats['correct'] / initial_has_correct_stats['total']
            if initial_has_correct_stats['total'] else 0
        ),
        'subq_accuracy_when_initial_no_correct': (
            initial_no_correct_stats['correct'] / initial_no_correct_stats['total']
            if initial_no_correct_stats['total'] else 0
        ),
        'main_stats': main_stats
    }

    return metrics

def print_metrics(metrics):
    """Красивый вывод метрик."""
    print("\n" + "="*60)
    print("ИТОГОВЫЕ МЕТРИКИ")
    print("="*60)
    print(f"Всего подвопросов: {metrics['total_subquestions']}")
    print(f"Точность на подвопросах (subq_accuracy): {metrics['subq_accuracy']:.2%}")
    print(f"Доля успешных завершений агента: {metrics['subq_success_rate']:.2%}")
    print(f"Среднее количество шагов на подвопрос: {metrics['avg_steps_per_subq']:.2f} ± {metrics['std_steps_per_subq']:.2f} "
          f"(мин: {metrics['min_steps_per_subq']}, макс: {metrics['max_steps_per_subq']})")
    print(f"\nВсего основных вопросов: {metrics['total_main_questions']}")
    print(f"Основные вопросы, где все подвопросы отвечены верно: {metrics['main_all_correct_count']} "
          f"({metrics['main_accuracy']:.2%})")
    print(f"\nТочность на подвопросах:")
    print(f"  - когда правильный документ был в начальной выдаче: {metrics['subq_accuracy_when_initial_has_correct']:.2%}")
    print(f"  - когда правильного документа не было в начальной выдаче: {metrics['subq_accuracy_when_initial_no_correct']:.2%}")

    print("\nДетализация по основным вопросам:")
    for main_id, stats in metrics['main_stats'].items():
        print(f"  Main {main_id}: {stats['num_correct']}/{stats['num_subq']} верных подвопросов "
              f"({'все верны' if stats['all_correct'] else 'не все'}) (main_correct={stats.get('main_correct', 'N/A')})")